# MAI511-2 Advanced ML - Lab 3: Feature Selection & Reduction
Linear Regression (target: *Life expectancy*) and Logistic Regression (target: *Status*, Developed=1).

Place `lab3_core.py` and `Life_Expectancy_Data.csv` next to this notebook (in Colab: upload both files, then run all cells).

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from lab3_core import *
pd.set_option('display.width',220); pd.set_option('display.max_columns',40)
df = load_data('Life_Expectancy_Data.csv'); df.shape

## Step 1 - Preprocess & explore (both tasks)

In [ ]:
P = {t: prepare(df, t) for t in (REG, CLF)}
for t,p in P.items():
    print(t, '| instances', p['n_instances'], '| features', p['n_features'], '| train/test', p['n_train'], p['n_test'])
    print(p['missing'][p['missing']>0].sort_values(ascending=False).to_string(), '\n')

## Steps 2-6 - run the whole pipeline for each task
`run_all` performs: feature selection (Low Variance, High Correlation, Factor Analysis, Backward, Forward) -> metrics + 5-fold CV -> gradient descent -> GridSearchCV -> comparison. It takes a few minutes (Forward/Backward + GridSearchCV).

In [ ]:
R = {t: run_all(df, t) for t in (REG, CLF)}

### Linear Regression

In [ ]:
r = R[REG]
for n, fs in r['sets'].items():
    print(f'{n} ({len(fs["features"])}):', fs['features'])

**Metrics + k-fold CV (mean, std)**

In [ ]:
R[REG]['eval'].round(4)

**Gradient descent: learning rate x iterations** (loss@N = training loss after N iterations)

In [ ]:
R[REG]['lr_table'].round(4)

In [ ]:
fig, ax = plt.subplots(1,2,figsize=(14,4.5))
for lr, res in R[REG]['lr_runs'].items():
    if not res['diverged']: ax[0].plot(res['history'], label=f'lr={lr}')
ax[0].set_xscale('log'); ax[0].set_yscale('log'); ax[0].legend(); ax[0].set_title('Loss vs iteration')
for n, h in R[REG]['gd_curves'].items(): ax[1].plot(h, label=n)
ax[1].set_xscale('log'); ax[1].set_yscale('log'); ax[1].legend(fontsize=7); ax[1].set_title('Convergence by feature set'); plt.show()
R[REG]['gd_sets'].round(4)

**GridSearchCV**

In [ ]:
R[REG]['grid'].round(4)

**Comparison**

In [ ]:
R[REG]['comp'].round(4)

### Logistic Regression

In [ ]:
r = R[CLF]
for n, fs in r['sets'].items():
    print(f'{n} ({len(fs["features"])}):', fs['features'])

**Metrics + k-fold CV (mean, std)**

In [ ]:
R[CLF]['eval'].round(4)

**Gradient descent: learning rate x iterations** (loss@N = training loss after N iterations)

In [ ]:
R[CLF]['lr_table'].round(4)

In [ ]:
fig, ax = plt.subplots(1,2,figsize=(14,4.5))
for lr, res in R[CLF]['lr_runs'].items():
    if not res['diverged']: ax[0].plot(res['history'], label=f'lr={lr}')
ax[0].set_xscale('log'); ax[0].set_yscale('log'); ax[0].legend(); ax[0].set_title('Loss vs iteration')
for n, h in R[CLF]['gd_curves'].items(): ax[1].plot(h, label=n)
ax[1].set_xscale('log'); ax[1].set_yscale('log'); ax[1].legend(fontsize=7); ax[1].set_title('Convergence by feature set'); plt.show()
R[CLF]['gd_sets'].round(4)

**GridSearchCV**

In [ ]:
R[CLF]['grid'].round(4)

**Comparison**

In [ ]:
R[CLF]['comp'].round(4)

## Step 7 - Self-learning component
(A) forward-selection k-sweep, (B) embedded selectors (Lasso / Random Forest), (C) improvement ladder (log transform -> KNN imputation -> polynomial + regularisation -> threshold tuning, plus a boosting benchmark), (D) leakage check with country-grouped CV.

In [ ]:
S = {t: self_learning(R[t]['prep']) for t in (REG, CLF)}

### Linear

In [ ]:
s = S[REG]
print('suggested k =', s['k_sweep'].attrs['suggested_k'])
print('Lasso keeps:', s['embedded']['lasso'])
print('RF top-10:', s['embedded']['rf_top'])
s['extra_eval'].round(4)

In [ ]:
S[REG]['ladder'].round(4)

In [ ]:
S[REG]['leakage'].round(4)

### Logistic

In [ ]:
s = S[CLF]
print('suggested k =', s['k_sweep'].attrs['suggested_k'])
print('Lasso keeps:', s['embedded']['lasso'])
print('RF top-10:', s['embedded']['rf_top'])
s['extra_eval'].round(4)

In [ ]:
S[CLF]['ladder'].round(4)

In [ ]:
S[CLF]['leakage'].round(4)

## Observations
See README.md for the written observations and viva notes; every number above is produced by the code (nothing is hard-coded).